# Model Evaluation — Readmission Risk Prediction

Detailed evaluation of the champion registered model: confusion matrix,
classification report, feature importance.

**Reads:** `gold_ml_features`, `gold_ml_scoring_meta`, MLflow registry

**Writes:** `gold_ml_feature_importance`

In [ ]:
import json
import mlflow
import mlflow.lightgbm
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp

spark = SparkSession.builder.getOrCreate()
print('Spark session ready')

In [ ]:
# Load champion model from the MLflow model registry (latest version).
# Flavor-aware load returns the native sklearn-API model (predict_proba available).
meta = json.loads(spark.read.table('gold_ml_scoring_meta').collect()[0]['meta_json'])
champion_name = meta['champion_model']
champion_flavor = meta['champion_flavor']
feature_cols = meta['feature_cols']
numeric_features = meta['numeric_features']
cat_cols = meta['cat_cols']
category_maps = meta['category_maps']
LABEL = meta['label']

uri = f'models:/{champion_name}/latest'
model = mlflow.lightgbm.load_model(uri) if champion_flavor == 'lightgbm' else mlflow.sklearn.load_model(uri)
print(f'Loaded registered model: {champion_name} ({champion_flavor}, latest version)')

In [ ]:
# Rebuild the same pandas feature frame as training (named float64 columns)
select_cols = numeric_features + cat_cols + [LABEL]
pdf = spark.read.table('gold_ml_features').select(select_cols).toPandas()
for c in numeric_features:
    pdf[c] = pd.to_numeric(pdf[c], errors='coerce').fillna(0.0)
for c in cat_cols:
    pdf[f'{c}_idx'] = pdf[c].astype(str).map(category_maps[c]).fillna(-1).astype('float64')

X = pdf[feature_cols].astype('float64')
y = pdf[LABEL].astype('int64')

# Same split + seed as training, so this is the same held-out validation set
_, X_val, _, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
y_pred = model.predict(X_val)
proba = model.predict_proba(X_val)[:, 1]
print(f'Test predictions: {len(y_pred):,} rows')

In [ ]:
# Confusion matrix + classification report on the held-out set
print('=== Confusion Matrix ===')
print(confusion_matrix(y_val, y_pred))
print('\n=== Classification Report ===')
print(classification_report(y_val, y_pred, digits=3))
print(f'AUC-ROC: {roc_auc_score(y_val, proba):.4f}')

In [ ]:
# Feature importance — named columns align 1:1 with feature_cols
imp = np.asarray(model.feature_importances_, dtype='float64')
imp = imp / imp.sum() if imp.sum() > 0 else imp
rows = sorted(zip(feature_cols, [float(v) for v in imp]), key=lambda r: r[1], reverse=True)

print('=== Top 10 Features by Importance ===')
for name, v in rows[:10]:
    print(f'  {name:30s} {v:.4f}')

fi_spark = (
    spark.createDataFrame(rows, ['feature', 'importance'])
    .withColumn('model_timestamp', current_timestamp())
)
fi_spark.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_feature_importance')
print('Feature importance saved to gold_ml_feature_importance')

In [ ]:
spark.sql('OPTIMIZE gold_ml_feature_importance')
print('Evaluation complete')